# Connected hypergraphs: LOMO × Merkle break points × MMR (replayable)

Nodes are content addresses; hyperedges (custody, recipe, amendment, truth, toy, evaluation, consensus, execution, ledger) join many nodes and are hashed. The hypergraph root covers every edge; snapshots are appended to `daisy/hypergraph_mmr.jsonl`. LOMO removes a node (here an actor) and recomputes what depends on it.

In [1]:
import os, sys, json
from pathlib import Path
if Path.cwd().name == 'notebooks': os.chdir('..')
sys.path.insert(0, os.getcwd())
from baebugs import hypergraph as HG
g, log = HG.snapshot('.')
print('root', g['root'][:16], g['counts'], 'nodes', len(g['nodes']), '| snapshots', len(log.records), log.verify())

root 0b8f707fa706c619 {'amendment': 2, 'consensus': 4, 'custody': 20, 'datasets_root': 1, 'evaluation': 52, 'execution': 1, 'ledger': 9, 'recipe': 18, 'toy': 91, 'truth': 1} nodes 406 | snapshots 2 (True, None)


## Walk any node back to the frozen truth

In [2]:
for start in ('dataset:multires_mp_1', 'protocol:DAISY-BB-001b'):
    print(start)
    for step in HG.walk_to_truth(g, start): print('   ', step[0], step[1], step[2][:60])

dataset:multires_mp_1
    datasets_root 81abcb94b5f0 datasets_root:5a0d1661538f2a9c1338a274f90477cade19ed034bb09b
    truth 93391532d984 truth:5d190ae6be4b3b6b4aaa5bf32ce030ad2a09d06ae4589e63e2aeb9
protocol:DAISY-BB-001b
    recipe 106817ecfcf5 dataset:synth_1_30_r2
    datasets_root 81abcb94b5f0 datasets_root:5a0d1661538f2a9c1338a274f90477cade19ed034bb09b
    truth 93391532d984 truth:5d190ae6be4b3b6b4aaa5bf32ce030ad2a09d06ae4589e63e2aeb9


## Leave-one-actor-out: which verifier's attestation changes the consensus

In [3]:
for v in ('exact', 'tol_0.5mm', 'plan_only'):
    r = HG.lomo_actors('.', v)
    print(v, r['baseline'])
    for a, x in list(r['leave_one_actor_out'].items())[:5]: print('   ', a, x['flips'], x['examples'][:2])

exact {'DISPUTED': 86, 'FAILED': 5}
    lfm2.5-1.2b-instruct_q4_k_m 0 []
    qwen3_1.7b 0 []
    classical@10px/mm 0 []
    classical@20px/mm 0 []
    classical@5px/mm 0 []
tol_0.5mm {'SUPPORTED': 15, 'DISPUTED': 72, 'FAILED': 4}
    lfm2.5-1.2b-instruct_q4_k_m 6 ['Formicidae_mean_6.9mm', 'Gerridae_mean-1SD_16.2mm']
    qwen3_1.7b 6 ['Formicidae_mean_6.9mm', 'Gerridae_mean-1SD_16.2mm']
    yolo@10px/mm 6 ['Formicidae_mean_6.9mm', 'Gerridae_mean-1SD_16.2mm']
    yolo@5px/mm 6 ['Formicidae_mean_6.9mm', 'Gerridae_mean-1SD_16.2mm']
    yoloe@5px/mm 4 ['Formicidae_mean_6.9mm', 'Halictidae_mean-2SD_2.9mm']
plan_only {'SUPPORTED': 18, 'DISPUTED': 69, 'FAILED': 4}
    yolo@10px/mm 8 ['Andrenidae_mean+2SD_13.6mm', 'Apidae_mean_13.5mm']
    yoloe@5px/mm 8 ['Andrenidae_mean+2SD_13.6mm', 'Apidae_mean_13.5mm']
    yolo@5px/mm 7 ['Andrenidae_mean+2SD_13.6mm', 'Apidae_mean_13.5mm']
    yoloe@20px/mm 7 ['Andrenidae_mean+2SD_13.6mm', 'Apidae_mean_13.5mm']
    lfm2.5-1.2b-instruct_q4_k_m 6 ['Cicadidae_m